# UTS SOAL 2 — Spark RDD (25 poin)
**Jalankan berurutan dengan kernel BigDataLab di JupyterLab Master.** Menggunakan snapshot Kafka **yang sama** dengan Soal 3 lewat `snapshot_offsets.json`. Tidak memerlukan drive `Z:` atau `C:\shared`.

### SEL 1 — konfigurasi

In [1]:
# WAJIB DIJALANKAN PERTAMA SETIAP KALI KERNEL DIRESTART. Tidak perlu config.py terpisah.
import os, sys, json, time, socket, datetime, importlib.util
from pathlib import Path
MASTER_IP = os.environ.get('BIGDATA_MASTER_IP', '192.168.1.32')  # terbaru: panel MASTER
KAFKA_BOOTSTRAP = os.environ.get('BIGDATA_KAFKA', MASTER_IP + ':9092')
KAFKA_TOPIC = os.environ.get('BIGDATA_TOPIC', 'wiki.recentchange.raw')
SPARK_MASTER = os.environ.get('BIGDATA_SPARK', 'spark://' + MASTER_IP + ':7077')
# Hasil/output disimpan pada komputer MASTER tempat kernel Jupyter berjalan.
DEFAULT_DIR = 'C:/BigData/work/UTS_Wikipedia_2026' if os.name == 'nt' else str(Path.cwd() / '_hasil_uts')
BASE = Path(os.environ.get('BIGDATA_OUTPUT_DIR', DEFAULT_DIR))
DATA, HASIL, STATE = BASE/'data', BASE/'hasil', BASE/'state'
for folder in (DATA,HASIL,STATE):folder.mkdir(parents=True,exist_ok=True)
MANIFEST_PATH = DATA/'snapshot_offsets.json'
print('KERNEL:',sys.executable)
print('HOST KERNEL:',socket.gethostname())
print('MASTER:',SPARK_MASTER,'| KAFKA:',KAFKA_BOOTSTRAP,'| TOPIC:',KAFKA_TOPIC)
print('FOLDER HASIL:',BASE)


KERNEL: C:\BigData\venv\Scripts\python.exe
HOST KERNEL: BIGDATA-MASTER
MASTER: spark://192.168.1.32:7077 | KAFKA: 192.168.1.32:9092 | TOPIC: wiki.recentchange.raw
FOLDER HASIL: C:\BigData\work\UTS_Wikipedia_2026


### SEL 2 — paket Kafka

In [2]:
# CEK PERPUSTAKAAN DAN KONEKSI. Jangan otomatis memasang ulang paket bawaan BigDataLab.
needed = {'kafka':'kafka-python','requests':'requests'}
missing = [package for module,package in needed.items() if importlib.util.find_spec(module) is None]
if missing:
    raise RuntimeError('Paket Python belum tersedia: '+', '.join(missing)+
      '. Jalankan DI KERNEL BigDataLab: %pip install '+ ' '.join(missing))
from kafka import KafkaConsumer, KafkaAdminClient, TopicPartition, KafkaProducer
from kafka.admin import NewTopic
from kafka.errors import TopicAlreadyExistsError
for port,label in [(7077,'Spark Master'),(9092,'Kafka')]:
    try:
        with socket.create_connection((MASTER_IP,port),timeout=4):
            print(label,port,'TCP TERJANGKAU (bukan jaminan metadata/broker valid).')
    except OSError as exc:
        print(label,port,'TCP GAGAL:',exc)
print('LIBRARY KAFKA TERSEDIA; langkah berikutnya mengecek metadata broker.')


Spark Master 7077 TCP TERJANGKAU (bukan jaminan metadata/broker valid).
Kafka 9092 TCP TERJANGKAU (bukan jaminan metadata/broker valid).
LIBRARY KAFKA TERSEDIA; langkah berikutnya mengecek metadata broker.


### SEL 3 — Spark Master + konektor Kafka

In [3]:
APP_NAME='UTS_S2_RDD_Cluster'

# Spark Standalone; driver berjalan di PC Master via JupyterLab BigDataLab.
# Pastikan kernel fresh ketika mengubah IP atau mengaktifkan konektor Spark-Kafka.
import pyspark
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType,StructField,LongType,IntegerType,StringType,BooleanType
SPARK_KAFKA_PACKAGE = 'org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.3'
if not pyspark.__version__.startswith('4.1.'):
    raise RuntimeError('PySpark='+pyspark.__version__+'; cek versi Spark sebelum mengubah paket konektor!')
existing=SparkSession.getActiveSession()
if existing is not None and existing.sparkContext.master != SPARK_MASTER:
    raise RuntimeError('Sudah ada SparkSession ke '+existing.sparkContext.master+
                       '. Restart KERNEL BigDataLab lalu jalankan dari sel 1; jangan ganti variabel saja.')
spark=(SparkSession.builder.master(SPARK_MASTER)
  .appName(APP_NAME)
  .config('spark.driver.host',MASTER_IP)
  .config('spark.driver.bindAddress','0.0.0.0')
  .config('spark.executor.memory','1g')
  .config('spark.cores.max','6')
  .config('spark.jars.packages', SPARK_KAFKA_PACKAGE)
  .config('spark.sql.shuffle.partitions','8')
  .config('spark.sql.session.timeZone','UTC')
  .getOrCreate())
sc=spark.sparkContext
print('SPARK:',spark.version,'| MASTER:',sc.master,'| UI:',sc.uiWebUrl)
if sc.master!=SPARK_MASTER:
    raise RuntimeError('Spark berjalan di alamat berbeda. Restart KERNEL; periksa IP Master.')


SPARK: 4.1.3 | MASTER: spark://192.168.1.32:7077 | UI: http://192.168.1.32:4040


### SEL 4 — manifest snapshot sama dengan Soal 3

In [5]:
# SNAPSHOT BERSAMA 2 & 3: rentang offset Kafka dibekukan SATU KALI.
# Menyimpan offset, bukan menyalin data privat ke GitHub.
from kafka import KafkaConsumer, TopicPartition
monitor = KafkaConsumer(bootstrap_servers=[KAFKA_BOOTSTRAP],group_id=None,
                        enable_auto_commit=False,request_timeout_ms=30000)
try:
    if MANIFEST_PATH.exists():
        manifest=json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
        print('SNAPSHOT LAMA DIPAKAI TANPA DIUBAH:',MANIFEST_PATH)
    else:
        parts=sorted(monitor.partitions_for_topic(KAFKA_TOPIC) or [])
        if not parts:raise RuntimeError('TOPIC Kafka belum ditemukan: '+KAFKA_TOPIC)
        tps=[TopicPartition(KAFKA_TOPIC,p) for p in parts]
        begins=monitor.beginning_offsets(tps);ends=monitor.end_offsets(tps)
        manifest={'topic':KAFKA_TOPIC,'master_ip_saat_dibekukan':MASTER_IP,
          'dibuat_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
          'start':{str(tp.partition):int(begins[tp]) for tp in tps},
          'end':{str(tp.partition):int(ends[tp]) for tp in tps}}
        manifest['total_offsets']=sum(manifest['end'][p]-manifest['start'][p] for p in manifest['end'])
        if manifest['total_offsets']<=0:raise RuntimeError('Topic belum berisi data untuk snapshot.')
        MANIFEST_PATH.write_text(json.dumps(manifest,indent=2),encoding='utf-8')
        print('SNAPSHOT BARU BEKU:',MANIFEST_PATH)
    if manifest['topic']!=KAFKA_TOPIC:
        raise RuntimeError('Topic pada snapshot berbeda! Periksa '+str(MANIFEST_PATH))
    print('PESAN DLM RENTANG OFFSET:',manifest['total_offsets'])
    print('OFFSET AWAL:',manifest['start'],'OFFSET AKHIR:',manifest['end'])
    if manifest['total_offsets']<200000:
        print('UTS 2: 200.000 ATAU seluruh data terkumpul. Jelaskan jumlah aktual dalam laporan.')
finally:monitor.close()


SNAPSHOT LAMA DIPAKAI TANPA DIUBAH: C:\BigData\work\UTS_Wikipedia_2026\data\snapshot_offsets.json
PESAN DLM RENTANG OFFSET: 453974
OFFSET AWAL: {'0': 0, '1': 0, '2': 0, '3': 0} OFFSET AKHIR: {'0': 121140, '1': 108101, '2': 113414, '3': 111319}


### SEL 5 — baca topic sebagai batch → RDD

In [6]:
# Baca Kafka secara batch pada OFFSET TETAP; tidak membutuhkan C:\shared pada Worker.
# Pastikan Spark Kafka connector sudah tersedia di JAR Spark 4.1.3.
startingOffsets=json.dumps({KAFKA_TOPIC:manifest['start']})
endingOffsets=json.dumps({KAFKA_TOPIC:manifest['end']})
try:
    raw_df=(spark.read.format('kafka')
      .option('kafka.bootstrap.servers',KAFKA_BOOTSTRAP)
      .option('subscribe',KAFKA_TOPIC)
      .option('startingOffsets',startingOffsets)
      .option('endingOffsets',endingOffsets)
      .option('failOnDataLoss','true')
      .load().select(F.col('value').cast('string').alias('raw_json')))
    print('READ KAFKA BATCH OK; DATA AKAN TERBACA SAAT ACTION .count()/.collect().')
except Exception as err:
    print('KONEKTOR SPARK-KAFKA GAGAL. Restart kernel sesudah memeriksa JAR. Error asli:')
    raise


READ KAFKA BATCH OK; DATA AKAN TERBACA SAAT ACTION .count()/.collect().


### SEL 6 — 2A: parsing aman, accumulator, glom & partisi

In [16]:
import re
from pyspark import StorageLevel
raw_rdd=raw_df.rdd.map(lambda row: row['raw_json'])
bad_acc=sc.accumulator(0)
def parse_record(line):
    try:
        d=json.loads(line)
        if isinstance(d,dict) and d.get('wiki'):
            return d
    except (ValueError,TypeError):pass
    bad_acc.add(1)
    return None
parsed=raw_rdd.map(parse_record).filter(lambda r:r is not None).persist(StorageLevel.MEMORY_AND_DISK)
valid=parsed.count() # MATERIALIZE sekali, cegah hitung accumulator berulang
part_sizes=parsed.glom().map(len).collect()
print('VALID:',valid,'RUSAK (acc):',bad_acc.value)
print('PARTISI:',parsed.getNumPartitions(),'SEBARAN glom():',part_sizes)
print('CATAT: retries task bisa menggandakan accumulator; lihat Spark UI untuk indikasi task retry.')


VALID: 453968 RUSAK (acc): 6
PARTISI: 4 SEBARAN glom(): [121140, 108095, 113414, 111319]
CATAT: retries task bisa menggandakan accumulator; lihat Spark UI untuk indikasi task retry.


In [12]:
import sys
import pyspark

print("=== DIAGNOSTIK SPARK ===")

print("Python:", sys.version)
print("Python Path:", sys.executable)
print("PySpark:", pyspark.__version__)
print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print("Python Executor:", spark.sparkContext.pythonExec)

print("\n=== TES SPARK ===")

try:
    hasil = spark.range(3).count()
    print("Tes Spark JVM: BERHASIL", hasil)
except Exception as e:
    print("Tes Spark JVM: GAGAL")
    print(str(e)[:500])

try:
    hasil = (
        spark.sparkContext
        .parallelize([1, 2, 3], 2)
        .map(lambda x: x + 1)
        .collect()
    )

    print("Tes Python RDD: BERHASIL", hasil)

except Exception as e:
    print("Tes Python RDD: GAGAL")
    print(str(e)[:500])

=== DIAGNOSTIK SPARK ===
Python: 3.14.8 (tags/v3.14.8:8e6e75d, Sep 30 2026, 18:19:33) [MSC v.1944 64 bit (AMD64)]
Python Path: C:\BigData\venv\Scripts\python.exe
PySpark: 4.1.3
Spark: 4.1.3
Master: spark://192.168.1.32:7077
Python Executor: C:\BigData\venv\Scripts\python.exe

=== TES SPARK ===


Tes Spark JVM: BERHASIL 3


Tes Python RDD: BERHASIL [2, 3, 4]


In [13]:
import sys

print("=== KONFIGURASI ===")
print("Python:", sys.version)
print("Python Path:", sys.executable)
print("Spark Master:", spark.sparkContext.master)
print("Python Executor:", spark.sparkContext.pythonExec)

print("\n=== TES SPARK ===")

try:
    hasil = spark.range(3).count()
    print("TES JVM: BERHASIL", hasil)
except Exception as e:
    print("TES JVM: GAGAL")
    print(str(e)[:500])

try:
    hasil = (
        spark.sparkContext
        .parallelize([1, 2, 3], 2)
        .map(lambda x: x + 1)
        .collect()
    )
    print("TES PYTHON RDD: BERHASIL", hasil)
except Exception as e:
    print("TES PYTHON RDD: GAGAL")
    print(str(e)[:1000])

=== KONFIGURASI ===
Python: 3.14.8 (tags/v3.14.8:8e6e75d, Sep 30 2026, 18:19:33) [MSC v.1944 64 bit (AMD64)]
Python Path: C:\BigData\venv\Scripts\python.exe
Spark Master: spark://192.168.1.32:7077
Python Executor: C:\BigData\venv\Scripts\python.exe

=== TES SPARK ===


TES JVM: BERHASIL 3


TES PYTHON RDD: BERHASIL [2, 3, 4]


### SEL 7 — 2B(i): top 10 wiki edit **manusia** (`filter`, `map` narrow; `reduceByKey` wide; `takeOrdered` action)

In [17]:
sc.setJobGroup('S2B_TOP10_WIKI','Top 10 wiki edit manusia')
human_pairs=(parsed.filter(lambda d:d.get('type')=='edit' and d.get('bot') is False and d.get('wiki'))
  .map(lambda d:(d['wiki'],1)))
top_wikis=human_pairs.reduceByKey(lambda a,b:a+b).takeOrdered(10,key=lambda t:(-t[1],t[0]))
print('TOP 10 WIKI:',top_wikis)
(HASIL/'s2_top10_wiki.json').write_text(json.dumps(top_wikis,indent=2,ensure_ascii=False),encoding='utf-8')


TOP 10 WIKI: [('wikidatawiki', 80784), ('enwiki', 17672), ('commonswiki', 17174), ('frwiki', 3863), ('dewiki', 3719), ('ruwiki', 2163), ('itwiki', 2082), ('jawiki', 1786), ('eswiki', 1464), ('zhwiki', 1343)]


336

### SEL 8 — 2B(ii): top 10 halaman di idwiki (edit saja)

In [18]:
sc.setJobGroup('S2B_TOP10_HALAMAN','Top 10 halaman idwiki')
page_counts=(parsed.filter(lambda d:d.get('wiki')=='idwiki' and d.get('type')=='edit' and d.get('title'))
 .map(lambda d:(d['title'],1)).reduceByKey(lambda a,b:a+b))
top_pages=page_counts.takeOrdered(10,key=lambda t:(-t[1],t[0]))
print('TOP 10 HALAMAN IDWIKI:',top_pages)


TOP 10 HALAMAN IDWIKI: [("D'Academy (musim 8)", 12), ('Marsada Band', 10), ('Matius Kamuntik', 10), ('Pengguna:Wong Langsep/Layout stasiun', 9), ('Suryimah', 7), ('Pengguna:LiškovBot/Switch', 6), ('Wikipedia:Intervensi pengurus terhadap vandalisme', 6), ('Emil Salim', 5), ('Daftar penghargaan dan nominasi yang diterima oleh Tiara Andini', 4), ('Herawati', 4)]


### SEL 9 — 2B(iii): top 15 kata komentar idwiki + stopword

In [19]:
STOPWORDS={'yang','dan','di','ke','dari','dalam','untuk','pada','dengan','atau',
 'ini','itu','tidak','ada','oleh','sebagai','juga','jadi','telah','sudah','sebuah',
 'edit','halaman','revisi','the','and','of','to','in','a','an','is','for','by','on'}
def words(comment):return [w for w in re.findall(r"[A-Za-zÀ-ÿ]{3,}",str(comment).lower()) if w not in STOPWORDS]
sc.setJobGroup('S2B_TOP15_KATA','Top 15 kata komentar idwiki')
word_rdd=(parsed.filter(lambda d:d.get('wiki')=='idwiki' and d.get('type')=='edit')
 .flatMap(lambda d:((w,1) for w in words(d.get('comment') or ''))))
top_words=word_rdd.reduceByKey(lambda a,b:a+b).takeOrdered(15,key=lambda t:(-t[1],t[0]))
print('TOP 15 KATA:',top_words)


TOP 15 KATA: [('altered', 84), ('iabot', 84), ('needing', 84), ('notification', 84), ('review', 84), ('sources', 84), ('suntingan', 51), ('massrollback', 50), ('bot', 47), ('istimewa', 38), ('kontribusi', 38), ('mengembalikan', 28), ('massal', 25), ('secara', 25), ('memperbaiki', 19)]


### SEL 10 — 2C: ukur `reduceByKey` vs `groupByKey().mapValues(sum)` (waktu nyata; data & aturan sama)
Gunakan Spark UI untuk bukti Shuffle Read/Write. Sel berikut mencoba mengambil statistik Job/Stage dari REST UI; bila UI tidak dapat diakses, screenshot manual.

In [20]:
import urllib.request
import urllib.error
benchmark_rows=[]
def run_bench(label,mode):
    sc.setJobGroup('BENCH_'+label,label)
    pairs=(parsed.filter(lambda d:d.get('type')=='edit' and d.get('bot') is False and d.get('wiki'))
       .map(lambda d:(d['wiki'],1)))
    t0=time.perf_counter()
    if mode=='reduce':res=pairs.reduceByKey(lambda a,b:a+b).collect()
    else:res=pairs.groupByKey().mapValues(sum).collect()
    dt=time.perf_counter()-t0
    row={'method':label,'seconds':round(dt,3),'distinct_wiki':len(res),'sum':sum(v for _,v in res)}
    benchmark_rows.append(row);print(row)
    return dict(res)
red=run_bench('reduceByKey','reduce')
grp=run_bench('groupByKey','group')
assert red==grp,'HASIL BERBEDA: periksa filter/operasi!'
(HASIL/'s2_benchmark.json').write_text(json.dumps(benchmark_rows,indent=2),encoding='utf-8')
print('KEDUA HASIL KONSISTEN. Simpan screenshot Spark UI > Stages (Shuffle Read/Write).')


{'method': 'reduceByKey', 'seconds': 12.806, 'distinct_wiki': 300, 'sum': 151892}


{'method': 'groupByKey', 'seconds': 16.38, 'distinct_wiki': 300, 'sum': 151892}
KEDUA HASIL KONSISTEN. Simpan screenshot Spark UI > Stages (Shuffle Read/Write).


### SEL 11 — 2D: Lineage, Job–Stage–Task dan Spark UI REST Shuffle Metrics

In [21]:
lineage=parsed.toDebugString()
print('LINEAGE:\n',lineage.decode('utf-8','replace') if isinstance(lineage,bytes) else lineage)
print('APP ID:',sc.applicationId,'UI:',sc.uiWebUrl)
tracker=sc.statusTracker()
actions=['S2B_TOP10_WIKI','S2B_TOP10_HALAMAN','S2B_TOP15_KATA','BENCH_reduceByKey','BENCH_groupByKey']
action_table=[]
for action in actions:
    job_ids=list(tracker.getJobIdsForGroup(action))
    stage_ids=set()
    for job_id in job_ids:
        info=tracker.getJobInfo(job_id)
        if info is not None:stage_ids.update(info.stageIds)
    tasks=0
    for sid in sorted(stage_ids):
        sinfo=tracker.getStageInfo(sid)
        if sinfo is not None:tasks+=int(sinfo.numTasks)
    row={'action_group':action,'jobs':len(job_ids),'stages':len(stage_ids),
         'tasks_planned':tasks,'stage_ids':sorted(stage_ids)}
    print('JOB-STAGE-TASK:',row)
    action_table.append(row)
(HASIL/'s2_job_stage_task.json').write_text(json.dumps(action_table,indent=2),encoding='utf-8')
# Ambil metrik stage aktual dari Spark UI REST, jika bisa diakses.
import urllib.request
if sc.uiWebUrl:
    url=sc.uiWebUrl.rstrip('/')+'/api/v1/applications/'+sc.applicationId+'/stages'
    try:
        with urllib.request.urlopen(url,timeout=8) as h:stages=json.load(h)
        summary=[{'stage':r.get('stageId'),'attempt':r.get('attemptId'),
          'numTasks':r.get('numTasks'),'shuffleReadBytes':r.get('shuffleReadBytes'),
          'shuffleWriteBytes':r.get('shuffleWriteBytes'),'name':r.get('name')}
          for r in stages]
        print('STAGE TERKUMPUL:',len(summary))
        for row in summary[-20:]:print(row)
        (HASIL/'s2_stages_rest.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
    except Exception as exc:
        print('REST UI TIDAK DAPAT DIAKSES; screenshot Job/Stage/Task manual.',str(exc)[:180])
else:print('Spark UI belum tersedia.')


LINEAGE:
 (4) PythonRDD[47] at RDD at PythonRDD.scala:59 [Disk Memory Serialized 1x Replicated]
 |       CachedPartitions: 4; MemorySize: 40.2 MiB; DiskSize: 0.0 B
 |  MapPartitionsRDD[10] at javaToPython at NativeMethodAccessorImpl.java:0 [Disk Memory Serialized 1x Replicated]
 |  MapPartitionsRDD[9] at javaToPython at NativeMethodAccessorImpl.java:0 [Disk Memory Serialized 1x Replicated]
 |  SQLExecutionRDD[8] at javaToPython at NativeMethodAccessorImpl.java:0 [Disk Memory Serialized 1x Replicated]
 |  MapPartitionsRDD[7] at javaToPython at NativeMethodAccessorImpl.java:0 [Disk Memory Serialized 1x Replicated]
 |  MapPartitionsRDD[6] at javaToPython at NativeMethodAccessorImpl.java:0 [Disk Memory Serialized 1x Replicated]
 |  MapPartitionsRDD[5] at javaToPython at NativeMethodAccessorImpl.java:0 [Disk Memory Serialized 1x Replicated]
 |  SQLExecutionRDD[4] at javaToPython at NativeMethodAccessorImpl.java:0 [Disk Memory Serialized 1x Replicated]
 |  MapPartitionsRDD[3] at javaToPython

STAGE TERKUMPUL: 30
{'stage': 19, 'attempt': 0, 'numTasks': 4, 'shuffleReadBytes': 0, 'shuffleWriteBytes': 0, 'name': 'collect at C:\\Users\\ASUS\\AppData\\Local\\Temp\\ipykernel_3104\\655573633.py:15'}
{'stage': 18, 'attempt': 0, 'numTasks': 4, 'shuffleReadBytes': 0, 'shuffleWriteBytes': 0, 'name': 'count at C:\\Users\\ASUS\\AppData\\Local\\Temp\\ipykernel_3104\\655573633.py:14'}
{'stage': 17, 'attempt': 0, 'numTasks': 2, 'shuffleReadBytes': 0, 'shuffleWriteBytes': 0, 'name': 'collect at C:\\Users\\ASUS\\AppData\\Local\\Temp\\ipykernel_3104\\3961958608.py:11'}
{'stage': 16, 'attempt': 0, 'numTasks': 4, 'shuffleReadBytes': 10623, 'shuffleWriteBytes': 0, 'name': 'takeOrdered at C:\\Users\\ASUS\\AppData\\Local\\Temp\\ipykernel_3104\\4165376723.py:4'}
{'stage': 15, 'attempt': 0, 'numTasks': 4, 'shuffleReadBytes': 0, 'shuffleWriteBytes': 10623, 'name': 'reduceByKey at C:\\Users\\ASUS\\AppData\\Local\\Temp\\ipykernel_3104\\4165376723.py:4'}
{'stage': 14, 'attempt': 0, 'numTasks': 2, 'shuffl

### SEL 12 — 2D: ukur cache vs tanpa cache (action yang sama); ambil bukti waktu

In [25]:
def parse_without_acc(line):
    try:
        d=json.loads(line)
        return d if isinstance(d,dict) and d.get('wiki') else None
    except (ValueError,TypeError):return None
uncached=raw_rdd.map(parse_without_acc).filter(lambda d:d is not None)
t0=time.perf_counter();c1=uncached.count();cold=time.perf_counter()-t0
cached=raw_rdd.map(parse_without_acc).filter(lambda d:d is not None).persist(StorageLevel.MEMORY_AND_DISK)
cached.count()  # warmup sebelum ukur action dari cache
t1=time.perf_counter();c2=cached.count();warm=time.perf_counter()-t1
assert c1==c2
print('JUMLAH:',c1,'TANPA CACHE (detik):',round(cold,3),'DENGAN CACHE (detik):',round(warm,3))
print('Perbandingan dipengaruhi load cluster; gunakan Spark UI untuk Job/Stage/Task.')
cached.unpersist()


JUMLAH: 453968 TANPA CACHE (detik): 220.024 DENGAN CACHE (detik): 6.321
Perbandingan dipengaruhi load cluster; gunakan Spark UI untuk Job/Stage/Task.


PythonRDD[81] at RDD at PythonRDD.scala:59

### SEL 13 — audit `wiki:id` logis; ekspor hasil untuk laporan

In [26]:
duplicates=(parsed.filter(lambda d:d.get('id') is not None)
 .map(lambda d:((str(d.get('wiki')),str(d.get('id'))),1))
 .reduceByKey(lambda a,b:a+b).filter(lambda kv:kv[1]>1).map(lambda kv:kv[1]-1).sum())
result={'valid_records':valid,'bad_accumulator':bad_acc.value,
   'partition_distribution':part_sizes,'top_wikis':top_wikis,
   'top_pages_idwiki':top_pages,'top_words_idwiki':top_words,
   'logical_duplicate_wiki_id':duplicates,'benchmark':benchmark_rows}
(HASIL/'soal2_hasil.json').write_text(json.dumps(result,indent=2,ensure_ascii=False),encoding='utf-8')
print('HASIL DISIMPAN KE:',HASIL/'soal2_hasil.json')


HASIL DISIMPAN KE: C:\BigData\work\UTS_Wikipedia_2026\hasil\soal2_hasil.json


In [24]:
if "cached" in globals():
    cached.unpersist(blocking=True)

print("Cache percobaan sebelumnya dibersihkan")

Cache percobaan sebelumnya dibersihkan
